# Projeto - Simulando a Opinião Pública
## Tema: Democracia
## Modelo escolhido: Gemma 3, 1B de parâmetros

## Primeiros passos
### Antes de executar as células de código, siga os passos a seguir
**Baixar e preparar Ollama:**
1. Escolha o comando apropriado para o seu SO (MacOS, Linux, Windows) no site do [Ollama](https://ollama.com/download)
> Para Linux/MacOS, pode ser necessário executar o comando abaixo
> ```bash
>  sudo apt-get install zstd
> ```
2.  Execute o servidor do Ollama localmente, o que deve abrir a porta `localhost:11434`
    ```bash
    ollama serve
    ```
> OBS: Se retornar que a porta já está em uso, é porque após o download, já foi executado automaticamente o comando anterior. Para confirmar, acesse `http://localhost:11434` no navegador e veja se retorna *Ollama is running* 
3. Execute o comando abaixo para baixar e executar o modelo:
    ```bash
    ollama run gemma3:1b "preload" > /dev/null
    ```
> OBS: Se não funcionar, execute `ollama pull gemma3:1b antes`

**Preparar ambiente virtual do Python:**<br><br>
Para baixar as dependências apenas para esse projeto (e não de forma fixa na sua máquina) execute essas instruções:
```bash
python3 -m venv .venv;
source .venv/bin/activate;
```

### Baixando bibliotecas necessárias

* **Pandas:** Para trabalhar com a base de dados
* **PyReadStat:** Para trabalhar com a base de dados, permitindo ler arquivo .sav e complementar o pandas
* **Ollama:** API do LLM
* **Pydantic + Typing**: Validação de formato de saída JSON

In [145]:
# Instalar a lib de Python para Ollama
!pip install ollama

In [146]:
# Instalar pndas
!pip install pandas

In [147]:
# Instalar a lib para ler arquivos .sav
!pip install pyreadstat

In [148]:
!pip install pydantic

### Desenvolvimento da conversa com o modelo

#### Parte 1) Selecionar variáveis mais relevantes
O primeiro passo é pedir ao LLM reconhecer quais são as variáveis mais relevantes para ele na hora de simular o participante.

Para isso, para cada pergunta, será enviado um prompt indicando essa tarefa, as variáveis com valores possíveis, a pergunta e um JSON Schema de resposta esperada.

##### Entrada
**Prompt de Sistema**
> "*Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
>
> *Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
> *Características possíveis: <lista de características, ignorando as outras perguntas>*

**Prompt de Entrada**
> *Pergunta: ...,Respostas Possíveis: [...], Múltipla Escolhda: Sim | Não*


##### Saída
```json
  {
    "question" : {
      "type": "string"
    },
    "caracteristicas_selecionadas": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "minItems": 1,
        "uniqueItems": true,
        "default": [
          "Nenhuma é relevante"
        ]
      },
  }
```


In [149]:
import pandas as pd

df = pd.read_spss("04832.SAV")

df.head()

,SEXO,IDADE,FX_ID,ESCOLARIDADE,P1A,P1B,P1C,P2_1,P2_2,P2_3,...,RACA,RELIGIAO,REND1,REND2,REGIAO,COND,PORTE,ID_Ipec,DATA_ENTREVISTA,TIPO_COLETA
0,FEM,29.0,25 A 34,Superior completo,Sim,Sim,Sim,Reduzir as desigualdades sociais,Reduzir a violência,Melhorar a qualidade da saúde,...,Branca,Outras Evangélicas específicas,MAIS DE 2 A 5,MAIS DE 10 A 20,NORDESTE,CAPITAL,MAIS DE 500.000,189519187.0,2023-09-01,Face a face
1,MAS,77.0,65 E MAIS,1ª série (ou 2º ano),Não,Não,Não,Reduzir a violência,Melhorar a qualidade da saúde,Melhorar a qualidade da educação,...,Parda,Católica Apostólica Romana,MAIS DE 2 A 5,MAIS DE 2 A 5,CENTRO OESTE,CAPITAL,MAIS DE 500.000,189525808.0,2023-09-01,Face a face
2,FEM,46.0,45 A 54,3ª série,Não,Não,Não,Reduzir as desigualdades sociais,Melhorar a qualidade da saúde,Combater as mudanças climáticas/desmatamento,...,Parda,Outras Evangélicas específicas,ATÉ 1,ATÉ 1,SUL,PERIFERIA,DE 100.001 A 500.000,189526277.0,2023-09-01,Face a face
3,MAS,72.0,65 E MAIS,3ª série (ou 4º ano),Não,Não,Não,Reduzir as desigualdades sociais,"Combater o preconceito (racismo, homofobia, di...",Incentivar a geração de empregos,...,Parda,Católica Apostólica Romana,ATÉ 1,ATÉ 1,SUDESTE,INTERIOR,DE 20.001 A 50.000,189532152.0,2023-09-01,Face a face
4,FEM,53.0,45 A 54,Superior completo,Não,Não,Não,Incentivar a geração de empregos,Melhorar a qualidade da saúde,Melhorar a qualidade da educação,...,Parda,Evangélica - Não sabe especificar,NÃO TEM RENDIMENTO PESSOAL,ATÉ 1,SUDESTE,INTERIOR,DE 20.001 A 50.000,189532154.0,2023-09-01,Face a face


In [150]:


#Investigando as características dos participantes e seus valores únicos
print("Colunas possíveis:", list(df.columns))

features_ignore: list[str] = ["ID_Ipec", "DATA_ENTREVISTA", "TIPO_COLETA","IDADE","REND1","REND2"]

#Excluindo as perguntas e características irrelevantes
feature_renda: pd.Series = pd.concat([df['REND1'].astype(str), df['REND2'].astype(str)], ignore_index=True).rename('RENDA')
participant_features: list[str] = list(df.columns[~df.columns.str.contains(r"P\d\D*")])
participant_features = [feature for feature in participant_features if feature not in features_ignore]
participant_features_values: dict[str, list[str]] = {feature: list(df[feature].unique()) for feature in participant_features}
participant_features.append(str(feature_renda.name))
participant_features_values['RENDA'] = list(feature_renda.unique())
print("Características dos participantes:", participant_features)
print("Valores únicos para cada característica:")
for feature, values in participant_features_values.items():
    print(f"  {feature}: {values}")

Colunas possíveis: ['SEXO', 'IDADE', 'FX_ID', 'ESCOLARIDADE', 'P1A', 'P1B', 'P1C', 'P2_1', 'P2_2', 'P2_3', 'P3_1', 'P3_2', 'P3_3', 'P3_4', 'P3_5', 'P3_6', 'P4', 'RACA', 'RELIGIAO', 'REND1', 'REND2', 'REGIAO', 'COND', 'PORTE', 'ID_Ipec', 'DATA_ENTREVISTA', 'TIPO_COLETA']
Características dos participantes: ['SEXO', 'FX_ID', 'ESCOLARIDADE', 'RACA', 'RELIGIAO', 'REGIAO', 'COND', 'PORTE', 'RENDA']
Valores únicos para cada característica:
  SEXO: ['FEM', 'MAS']
  FX_ID: ['25 A 34', '65 E MAIS', '45 A 54', '55 A 64', '35 A 44', '18 A 24', '16 E 17']
  ESCOLARIDADE: ['Superior completo', '1ª série (ou 2º ano)', '3ª série', '3ª série (ou 4º ano)', '4ª série (ou 5º ano)', '5ª série (ou 6º ano)', 'Superior incompleto', '2ª série (ou 3º ano)', '2ª série', 'Sabe ler/ escrever, mas não cursou escola', '8ª série (ou 9º ano)', 'Analfabeto', '1ª série', '7ª série (ou 8º ano)', '6ª série (ou 7º ano)', 'Pré-escola (ou 1º ano)']
  RACA: ['Branca', 'Parda', 'Preta', 'Indígena', 'Amarela']
  RELIGIAO: ['Out

In [151]:
questions = [
        {
            "code": "P11",
            "question": "Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?",
            "possible_answers": ["Sim", "Não", "Não Votou"],
            "multiple_choice": False
        },
        {
            "code": "P12",
            "question": "Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?",
            "possible_answers": ["Sim", "Não", "Não Votou"],
            "multiple_choice": False
        },
        {
            "code": "P13",
            "question": "Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?",
            "possible_answers": ["Sim", "Não", "Não Votou"],
            "multiple_choice": False
        },
        {
            "code": "P3",
            "question": " Algumas pessoas dizem que a divulgação de fake news, ou seja, de notícias ou conteúdos falsos podem prejudicar a democracia. "
                        "Pensando nisso, quais dessas opções você acredita que poderiam contribuir no combate à divulgação de fake news? ",
            "possible_answers": [
                "Ampliar a regulamentação, as regras a serem cumpridas pelas plataformas digitais (empresas de tecnologia como Facebook, Youtube, WhatsApp, Twitter/X, etc.)",
                "Responsabilizar e punir empresas de tecnologia e de comunicação que não removerem postagens com notícias ou conteúdos falsos",
                "Ampliar a regulamentação, as regras a serem cumpridas pelos usuários que divulgam ou compartilham fake news, criadas por eles próprios ou por terceiros",
                "Responsabilizar e punir os usuários que divulgam ou compartilham postagens com notícias ou conteúdos falsos",
                "Ampliar a regulamentação, as regras a serem cumpridas por políticos/candidatos que divulgam ou compartilham fake news, criadas por eles próprios ou por terceiros",
                "Responsabilizar, punir ou cassar políticos/candidatos que divulgam ou compartilham postagens com notícias ou conteúdos falsos",
                "Não sabe",
                "Não respondeu"
            ],
            "multiple_choice": True
        }
]

In [152]:
from typing import Set, Literal
from pydantic import BaseModel, Field

class ExpectedAnswerVariables(BaseModel):
    features_selected: Set[Literal["SEXO", "FX_ID", "ESCOLARIDADE", "RACA", "RENDA", "REGIAO", "RELIGIAO"]] = Field(
        min_length=1,
        description="Lista de características selecionadas pelo modelo, que devem ser as mais relevantes para responder a pergunta.",

    )

In [153]:
import ollama
import json


#Definindo Host do Ollama
ollama_host = 'http://localhost:11434'
ollama.Client(host=ollama_host)

participant_features_prompt = """
SEXO: Masculino, Feminino
FX_ID: Faixa de idade do participante, em décadas 
ESCOLARIDADE: Grau de escolaridade do participante
RACA: Raça do participante
RENDA: Faixa de renda mensal do participante, em milhares de reais
REGIAO: Região geográfica do país onde o participante reside
RELIGIAO: Religião do participante
COND: Se o participante vive na Capital, Interior ou Periferia
PORTE: Patrimônio financeiro do participante, em milhares de reais
"""

# Usando prompt de sistema
system_prompt = f"""
Você é um assistente de raciocínio treinado para entender como características pessoais moldam crenças. Sua tarefa é analisar características e selecionar as mais preditivas para o tópico de Democracia.*
Você receberá uma pergunta, suas possíveis respostas e se é de múltipla escolha ou escolha única, e deverá retornar uma lista de características que julgar que são as mais relevantes para responder a pergunta.*
Características possíveis: \n{participant_features_prompt}
IMPORTANTE: 
- RESPONDA APENAS COM O NOME DA CARACTERÍSTICA SELECIONADA, o qual está em LETRAS CAPITAIS na lista acima.
- Você deve APENAS RETORNAR A LISTA DE CARACTERÍSTICAS SELECIONADAS, sem explicações ou justificativas. 
- NÃO adicione nenhuma caracterísitca que não esteja na lista.
- NÃO adicione acentuação às características, mesmo que estejam acentuadas na lista. Por exemplo, escreva "RACA" ao invés de "RAÇA", "REGIAO" ao invés de "REGIÃO", e "RELIGIAO" ao invés de "RELIGIÃO".
"""

features_selected_for_question: dict[str, list[str]] = {}

# Chama da API do Ollama
for question in questions:
    user_question: str = f"""
    Pergunta: {question['question']}
    Possíveis respostas: {', '.join(question['possible_answers'])}
    Múltipla escolha: {'Sim' if question['multiple_choice'] else 'Não'}
    """
    try:
        response = ollama.chat(
            model='gemma3:1b',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_question},
            ],
            format=ExpectedAnswerVariables.model_json_schema(),
            options={"temperature": 0.3}
        )
        print("#" * 50)
        print(f"Pergunta Lida: {question['question']}")
        print("Resposta do Ollama:\n")
        features_selected_for_question[question['code']] = [feature.strip().upper() for feature in json.loads(response['message']['content'])['features_selected']]
        print(f"Características selecionadas: {features_selected_for_question[question['code']]}")
        print("#" * 50)
    except Exception as e:
        print(f"Erro ao obter resposta: {e}")

##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?
Resposta do Ollama:

Características selecionadas: ['SEXO', 'RELIGIAO']
##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

Características selecionadas: ['ESCOLARIDADE']
##################################################
##################################################
Pergunta Lida: Você se lembra em quem votou para Senador(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

Características selecionadas: ['SEXO', 'ESCOLARIDADE', 'RACA', 'RENDA', 'REGIAO']
##################################################
##################################################
Pergunta Lida:  Algumas pessoas dizem que a divulgação de fake news, ou seja, de notícias ou conteúdos fal

#### Parte 2) Simular participante
O segundo passo é pedir ao LLM responder às perguntas.

Para isso, para cada pergunta, será enviado um prompt indicando essa tarefa, o perfil do respondente e a pergunta

**Devem ser testados vários participantes reais. (TODO)**

##### Entrada
**Prompt de Sistema**
> "*Você é um participante de um questionário sobre Democracia. Sua tarefa é responder a pergunta fornecida, considerando o perfil passdo do participante que você está
> representando.*
> *IMPORTANTE: Responda apenas com a alternativa escolhida (ou com as alternativas, se for de múltipla escolha). NUNCA responda com uma alternativa que não esteja na lista

**Prompt de Entrada**
> *Perfil: [...]*<br>
>  *Pergunta: ...,Respostas Possíveis: [...], Múltipla Escolhda: Sim | Não*

##### Saída
**Escolha única**
```json
  {
    "answer": {
        "type": "string"
    }
  }
```

**Múltipla escolha**
```json
  {
    "answers": {
        "type": "array",
        "items": {
          "type": "string"
        },
        "minItems": 1,
        "uniqueItems": true,
      },
  }
```


### TODO: Modificar a entrada para receber um código para cada resposta possível, seguindo a lista abaixo:
Alternativas respostas esperadas: 1 a N, sendo N o número da última alternativa
Alternativas de ausência de resposta: 90 a N, sendo N a última alternativa
Ex:
Sim = 1
Não = 2
Não Votou = 3 (É uma reposta possível e esperada)
Não Respondeu = 90

**Objetivo:** O dataset usa números para as respostas, então para avaliar com as métricas, devem ser codificadas as respostas


In [154]:
class ExpectedSingleAnswer(BaseModel):
    answer: str = Field(
        description="Resposta selecionada pelo modelo, que deve ser a mais relevante para responder a pergunta"
    )

In [155]:
class ExpectedMultipleAnswer(BaseModel):
    answers: Set[str] = Field(
        min_length=1,
        description="Lista de respostas selecionadas pelo modelo, que devem ser as mais relevantes para responder a pergunta. As respostas devem ser escolhidas entre as opções de resposta fornecidas para cada pergunta."
    )

In [156]:
import re

def sort_renda(renda_unique_values: list[str]) -> list[str]:
    """
    Ordena as faixas de renda de forma lógica, considerando os termos "ATÉ", "MAIS DE" e "NÃO...".
   
    Args:
        renda_unique_values: Lista de faixas de renda em formato string, como "ATÉ
    
    Returns:
        Lista de faixas de renda ordenada logicamente.
    """
    def sort_key(s):

        #Extraindo número da palavra
        nums = list(map(int, re.findall(r'\d+', s)))
        
        if "ATÉ" in s:
            return (0, nums[0])  # menor prioridade, representa intervalo iniciado em 0
        
        elif "MAIS DE" in s:
            if len(nums) == 1:
                return (1, nums[0]) # ex: MAIS DE 20
            else:
                return (1, nums[0], nums[1])  # ex: MAIS DE 2 A 5
        
        else:
            return (-1, float('inf'))  # casos tipo "NÃO..." são os menores, equivalem a 0
    return sorted(renda_unique_values, key=sort_key)

def generate_profile(line_idx: int, participant_features: list[str], df: pd.DataFrame) -> str:
    """
    Procura um perfil de participante baseado nas features e valores selecionadas.
    
    Args:
        line_idx: Índice da linha no DataFrame
        participant_features: Lista de características a incluir no perfil
        df: DataFrame com os dados dos participantes
    
    Returns:
        String formatada com o perfil do participante
    """
    # Pegar o registro correspondente ao índice
    random_record = df.iloc[line_idx]
    
    # Extrair apenas as características relevantes
    profile = {}
    for feature in participant_features:
        if feature != 'RENDA':
            profile[feature] = random_record[feature]
        else:
            renda = sort_renda([str(random_record['REND1']), str(random_record['REND2'])])
            profile[feature] = f"{renda[0]} a {renda[1]} mil reais"
    
    # Formatando o perfil para exibição
    profile_str = "\n".join([f"{car}: {valor}" for car, valor in profile.items()])
    
    return profile_str

In [ ]:
system_prompt = f"""
Você é um participante de um questionário sobre Democracia. Sua tarefa é responder à pergunta fornecida, considerando o perfil do participante que você está representando.
IMPORTANTE: Responda apenas com a alternativa escolhida (ou com as alternativas, se for de múltipla escolha). NUNCA responda com uma alternativa que não esteja na lista
"""

for question in questions:
    idx: int = 1 # TEMPORÁRIO: NECESSÁRIO TESTAR VÁRIOS PARTICIPANTES, ENTÃO DEVE SER SUBSTITUÍDO POR UM LOOP QUE VARIA O ÍNDICE
    profile: str = generate_profile(idx, participant_features, df)
    user_question: str = f"""
    Perfil do participante: {profile}
    Pergunta: {question['question']}
    Possíveis respostas: {', '.join(question['possible_answers'])}
    Múltipla escolha: {'Sim' if question['multiple_choice'] else 'Não'}
    """
    try:
        response = ollama.chat(
            model='gemma3:1b',
            messages=[
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_question},
            ],
            format=(ExpectedMultipleAnswer.model_json_schema() if question['multiple_choice'] else ExpectedSingleAnswer.model_json_schema()),
            options={"temperature": 0.3}
        )
        print("#" * 50)
        print(f"Perfil do participante para a pergunta {question['code']}:\n{profile}\n")
        print(f"Pergunta Lida: {question['question']}")
        print("Resposta do Ollama:\n")
        print(response['message']['content'])
        print("#" * 50)
    except Exception as e:
        print(f"Erro ao obter resposta: {e}")
        raise e


##################################################
Perfil do participante para a pergunta P11:
SEXO: MAS
FX_ID: 65 E MAIS
ESCOLARIDADE: 1ª série (ou 2º ano)
RACA: Parda
RELIGIAO: Católica Apostólica Romana
REGIAO: CENTRO OESTE
COND: CAPITAL
PORTE: MAIS DE 500.000
RENDA: MAIS DE 2 A 5 a MAIS DE 2 A 5 mil reais

Pergunta Lida: Você se lembra em quem votou para Deputado(a) Estadual nas eleições gerais de 2022?
Resposta do Ollama:

{ "answer": "Não" }
##################################################
##################################################
Perfil do participante para a pergunta P12:
SEXO: MAS
FX_ID: 65 E MAIS
ESCOLARIDADE: 1ª série (ou 2º ano)
RACA: Parda
RELIGIAO: Católica Apostólica Romana
REGIAO: CENTRO OESTE
COND: CAPITAL
PORTE: MAIS DE 500.000
RENDA: MAIS DE 2 A 5 a MAIS DE 2 A 5 mil reais

Pergunta Lida: Você se lembra em quem votou para Deputado(a) Federal nas eleições gerais de 2022?
Resposta do Ollama:

{
  "answer": "Não"
}
############################################

## Parte 3) Avaliar Desempenho

### F1-Score/Acurácia
### Distribuição das respostas
### Explicabilidade (Importância das variáveis)

In [158]:
# TODO